# Task 3 — Neural Machine Translation

This notebook develops and evaluates recurrent sequence-to-sequence models for English–Italian machine translation.

The main objectives are:

1. Load the aligned and preprocessed sentence pairs produced in Task 2.
2. Split the data into training, validation, and test sets, reserving 20% for testing.
3. Develop an RNN-based encoder–decoder model for English-to-Italian translation.
4. Tune a small number of relevant hyperparameters using validation performance.
5. Evaluate the generated translations using BLEU and METEOR.
6. Reverse the translation direction and train an Italian-to-English model.
7. Develop a character-based model and compare it with the word-based models.

The implementation is intentionally kept simple, reproducible, and suitable for CPU execution.

## Evaluation metrics

Following the evaluation methods presented in the course material, this project uses **BLEU** and **METEOR** to evaluate the generated translations.

- **BLEU** measures the overlap of word n-grams between the generated translation and the reference translation. It also applies a brevity penalty to translations that are too short. BLEU will be computed over the complete test set, as recommended in the course material. A higher BLEU score indicates better translation quality.

- **METEOR** compares generated and reference translations using unigram precision and recall, while also considering the alignment and fragmentation of matched words. It complements BLEU because it is less dependent on exact multi-word n-gram matches. A higher METEOR score also indicates better translation quality.

These metrics were selected because they are reference-based machine translation metrics discussed in the course. Using both provides a more complete evaluation than using BLEU alone. Training and validation loss will still be monitored during model development, but they will not be used as the final translation-quality metrics.

In [13]:
# Imports
from pathlib import Path
import random

import numpy as np
import torch


# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# Device
DEVICE = torch.device("cpu")


# Project paths
PROJECT_ROOT = Path("..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "task3"
RESULTS_DIR = PROJECT_ROOT / "results" / "task3_outputs"

EN_PATH = PROCESSED_DIR / "task2_clean.en"
IT_PATH = PROCESSED_DIR / "task2_clean.it"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Check that the Task 2 outputs are available
if not EN_PATH.exists() or not IT_PATH.exists():
    raise FileNotFoundError(
        "Task 2 processed files were not found in data/processed. "
        "Complete Task 2 before running this notebook."
    )


print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"English data: {EN_PATH}")
print(f"Italian data: {IT_PATH}")

Project root: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-
Device: cpu
English data: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\data\processed\task2_clean.en
Italian data: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\data\processed\task2_clean.it


## 1. Data loading and splitting

The preprocessed English and Italian files produced in Task 2 are loaded as aligned sentence pairs. Each English sentence must remain paired with the Italian sentence on the same line.

The data is randomly divided using the fixed project seed:

- **70% training set:** used to train the models;
- **10% validation set:** used for hyperparameter selection;
- **20% test set:** used only for the final evaluation.

The same split will be reused for every experiment to ensure a fair comparison between translation directions, embedding approaches, and word-based and character-based models.

In [15]:
def read_lines(path):
    """Read a text file and return one sentence per line."""
    with path.open("r", encoding="utf-8") as file:
        return [line.rstrip("\n") for line in file]


english_sentences = read_lines(EN_PATH)
italian_sentences = read_lines(IT_PATH)

assert len(english_sentences) == len(italian_sentences), (
    "The English and Italian files are not aligned."
)

sentence_pairs = list(zip(english_sentences, italian_sentences))

print(f"Aligned sentence pairs: {len(sentence_pairs):,}")

#sanity check 
for i, (english, italian) in enumerate(sentence_pairs[:3], start=1):
    print(f"\nExample {i}")
    print(f"EN: {english}")
    print(f"IT: {italian}")

Aligned sentence pairs: 190,110

Example 1
EN: "madam president, on a point of order."
IT: "signora presidente, un richiamo al regolamento."

Example 2
EN: "madam president, coinciding with this year' s first part-session of the european parliament, a date has been set, unfortunately for next thursday, in texas in america, for the execution of a young 34 year-old man who has been sentenced to death. we shall call him mr hicks."
IT: "signora presidente, in coincidenza con la prima tornata dell'anno del parlamento europeo, negli stati uniti in texas è stata fissata, purtroppo per giovedì prossimo, l'esecuzione di un condannato a morte, un giovane di 34 anni che chiameremo di nome hicks."

Example 3
EN: "madam president, i would firstly like to compliment you on the fact that you have kept your word and that, during this first part-session of the new year, the number of television channels in our offices has indeed increased considerably."
IT: "signora presidente, mi permetta di farle inn

In [ ]:
TEST_RATIO = 0.20
VALIDATION_RATIO = 0.10
MAX_SENTENCE_LENGTH = 20    #maximum length of a sentence

sentence_pairs = [
    (english, italian)
    for english, italian in sentence_pairs
    if len(english.split()) <= MAX_SENTENCE_LENGTH
    and len(italian.split()) <= MAX_SENTENCE_LENGTH
]

print(f"Pairs after length filtering: {len(sentence_pairs):,}")

number_of_pairs = len(sentence_pairs)

number_of_test_pairs = int(number_of_pairs * TEST_RATIO)
number_of_validation_pairs = int(number_of_pairs * VALIDATION_RATIO)
number_of_training_pairs = (
    number_of_pairs
    - number_of_validation_pairs
    - number_of_test_pairs
)

rng = np.random.default_rng(SEED)
shuffled_indices = rng.permutation(number_of_pairs)

training_indices = shuffled_indices[:number_of_training_pairs]
validation_indices = shuffled_indices[
    number_of_training_pairs:
    number_of_training_pairs + number_of_validation_pairs
]
test_indices = shuffled_indices[
    number_of_training_pairs + number_of_validation_pairs:
]

training_pairs = [sentence_pairs[i] for i in training_indices]
validation_pairs = [sentence_pairs[i] for i in validation_indices]
test_pairs = [sentence_pairs[i] for i in test_indices]

print(f"Training pairs:   {len(training_pairs):,} ({len(training_pairs) / number_of_pairs:.1%})")
print(f"Validation pairs: {len(validation_pairs):,} ({len(validation_pairs) / number_of_pairs:.1%})")
print(f"Test pairs:       {len(test_pairs):,} ({len(test_pairs) / number_of_pairs:.1%})")

Pairs after length filtering: 70,914
Training pairs:   49,641 (70.0%)
Validation pairs: 7,091 (10.0%)
Test pairs:       14,182 (20.0%)


## 2. Word-level data preparation

The first translation model operates at the word level. Since Task 2 already lowercased the text and normalized whitespace, sentences are tokenized using a simple whitespace split. A more complex tokenizer would increase preprocessing complexity without being necessary for this baseline.

The following design choices are used:

- Sentences are limited to **20 tokens** in both languages before the train-validation-test split. This reduces training time and limits the long-term dependency problem of a simple RNN.
- After adding the start and end tokens, an encoded sentence can contain at most 22 tokens.
- The English and Italian vocabularies are built using only the training set, preventing information from the validation and test sets from influencing the model.
- Each vocabulary is limited to **5,000 tokens**, including the special tokens. The most frequent training words are retained, while less frequent words are represented by `<unk>`.
- `<sos>` and `<eos>` indicate the beginning and end of a sentence.
- `<pad>` is used to create batches of equal-length sequences.
- Padding is dynamic: each batch is padded only to its longest sentence rather than to the global maximum length.
- The original source lengths are retained so that the encoder can ignore padding when processing a batch.

These choices provide a simple word-based representation while keeping memory use and CPU training time under control. Their main limitation is that punctuation remains attached to words and rare words lose their identity through the `<unk>` token.

In [32]:
from collections import Counter


MAX_VOCABULARY_SIZE = 10_000

SPECIAL_TOKENS = ["<pad>", "<unk>", "<sos>", "<eos>"]
PAD_INDEX, UNK_INDEX, SOS_INDEX, EOS_INDEX = range(len(SPECIAL_TOKENS))


def tokenize(sentence):
    """Split a preprocessed sentence into words."""
    return sentence.split()


def build_vocabulary(sentences, max_size=MAX_VOCABULARY_SIZE):
    """Build a vocabulary from the most frequent training words."""
    word_counts = Counter(
        word
        for sentence in sentences
        for word in tokenize(sentence)
    )

    words = [
        word
        for word, _ in word_counts.most_common(
            max_size - len(SPECIAL_TOKENS)
        )
    ]

    index_to_token = SPECIAL_TOKENS + words
    token_to_index = {
        token: index
        for index, token in enumerate(index_to_token)
    }

    return token_to_index, index_to_token


english_training_sentences = [
    english for english, _ in training_pairs
]
italian_training_sentences = [
    italian for _, italian in training_pairs
]

english_token_to_index, english_index_to_token = build_vocabulary(
    english_training_sentences
)
italian_token_to_index, italian_index_to_token = build_vocabulary(
    italian_training_sentences
)

print(f"English vocabulary: {len(english_token_to_index):,} tokens")
print(f"Italian vocabulary: {len(italian_token_to_index):,} tokens")

English vocabulary: 10,000 tokens
Italian vocabulary: 10,000 tokens


In [33]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader


BATCH_SIZE = 128


def encode_sentence(sentence, token_to_index):
    """Convert a sentence into token indices."""
    word_indices = [
        token_to_index.get(word, UNK_INDEX)
        for word in tokenize(sentence)
    ]

    return torch.tensor(
        [SOS_INDEX] + word_indices + [EOS_INDEX],
        dtype=torch.long,
    )


class TranslationDataset(Dataset):
    """Store aligned source and target sentences."""

    def __init__(self, pairs, source_vocabulary, target_vocabulary):
        self.pairs = pairs
        self.source_vocabulary = source_vocabulary
        self.target_vocabulary = target_vocabulary

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        source_sentence, target_sentence = self.pairs[index]

        source = encode_sentence(
            source_sentence,
            self.source_vocabulary,
        )
        target = encode_sentence(
            target_sentence,
            self.target_vocabulary,
        )

        return source, target


def collate_batch(batch):
    """Pad the source and target sequences in one batch."""
    sources, targets = zip(*batch)

    source_lengths = torch.tensor(
        [len(source) for source in sources],
        dtype=torch.long,
    )

    padded_sources = pad_sequence(
        sources,
        batch_first=True,
        padding_value=PAD_INDEX,
    )
    padded_targets = pad_sequence(
        targets,
        batch_first=True,
        padding_value=PAD_INDEX,
    )

    return padded_sources, source_lengths, padded_targets


training_dataset = TranslationDataset(
    training_pairs,
    english_token_to_index,
    italian_token_to_index,
)
validation_dataset = TranslationDataset(
    validation_pairs,
    english_token_to_index,
    italian_token_to_index,
)
test_dataset = TranslationDataset(
    test_pairs,
    english_token_to_index,
    italian_token_to_index,
)

data_generator = torch.Generator().manual_seed(SEED)

training_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    generator=data_generator,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)


source_batch, source_lengths, target_batch = next(iter(training_loader))

print(f"Training examples: {len(training_dataset):,}")
print(f"Validation examples: {len(validation_dataset):,}")
print(f"Test examples: {len(test_dataset):,}")
print(f"Source batch shape: {source_batch.shape}")
print(f"Target batch shape: {target_batch.shape}")
print(f"First source lengths: {source_lengths[:5].tolist()}")

Training examples: 49,641
Validation examples: 7,091
Test examples: 14,182
Source batch shape: torch.Size([128, 22])
Target batch shape: torch.Size([128, 22])
First source lengths: [21, 8, 18, 13, 21]


## 3. Word-based RNN encoder–decoder

The model follows the encoder–decoder architecture presented in the course material:

1. The encoder reads the English sentence and compresses it into a hidden-state vector.
2. The decoder receives this vector and generates the Italian translation.
3. Encoder and decoder are trained jointly to maximize the probability of the correct target sentence.

A **one-layer vanilla RNN** is used for both components. This is the simplest architecture that directly satisfies the requirement of developing an RNN-based sequence-to-sequence model.

The vanilla RNN was selected instead of an LSTM or GRU mainly because the project requires several training experiments and all models must run on CPU. A simple RNN has fewer parameters and recurrent operations, reducing training time and keeping the implementation transparent.

The main architectural choices are:

- **Word-level input:** the character-based model is developed separately later.
- **One recurrent layer:** deeper networks would increase computational cost without being necessary for this baseline.
- **Unidirectional encoder:** it directly follows the basic sequential encoder described in the course.
- **64-dimensional embeddings:** a compact initial representation that limits the number of parameters.
- **128-dimensional hidden state:** large enough to represent short sentences while remaining computationally manageable.
- **Full teacher forcing during training:** the complete shifted target sequence can be processed efficiently in one operation.
- **Greedy decoding during inference:** it provides a simple and fast generation strategy.

Padding tokens are ignored by the encoder using packed sequences. The target padding tokens will later be ignored by the loss function.

A known limitation is that the complete English sentence is represented by a single fixed-size vector. In addition, vanilla RNNs may struggle with long-term dependencies because of the vanishing-gradient problem. The 20-token sentence limit reduces these problems and keeps training feasible on CPU. The impact of sentence length will be investigated in the final analysis.

- **Fixed context vector at every decoding step:** the final encoder hidden state initializes the decoder and is also concatenated with every target-word embedding. This creates a direct connection between the English sentence representation and every generated Italian word.
- **No attention:** the same fixed context vector is used at every step. The model does not compute dynamic weights or source-word alignments, so attention remains reserved for Task 4.

The initial baseline passed the encoder representation to the decoder only as its initial hidden state. A long training experiment showed that this configuration mainly learned common Italian Europarl sentences while making limited use of the English input. Reintroducing the same fixed context vector at every decoding step strengthens source conditioning without adding attention or a more expensive recurrent architecture.

In [48]:
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence


EMBEDDING_DIMENSION = 64
HIDDEN_DIMENSION = 128


class Encoder(nn.Module):
    """Encode a source sentence into its final hidden state."""

    def __init__(self, vocabulary_size, embedding_dimension, hidden_dimension):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dimension,
            padding_idx=PAD_INDEX,
        )
        self.rnn = nn.RNN(
            embedding_dimension,
            hidden_dimension,
            batch_first=True,
            nonlinearity="tanh",
        )

    def forward(self, source, source_lengths):
        embedded = self.embedding(source)

        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, hidden = self.rnn(packed)

        return hidden


class Decoder(nn.Module):
    """Generate target-word scores using the fixed encoder context."""

    def __init__(
        self,
        vocabulary_size,
        embedding_dimension,
        hidden_dimension,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dimension,
            padding_idx=PAD_INDEX,
        )

        self.rnn = nn.RNN(
            embedding_dimension + hidden_dimension,
            hidden_dimension,
            batch_first=True,
            nonlinearity="tanh",
        )

        self.output_layer = nn.Linear(
            hidden_dimension,
            vocabulary_size,
        )

    def forward(self, target_input, hidden, context):
        embedded = self.embedding(target_input)

        repeated_context = context[-1].unsqueeze(1).expand(
            -1,
            target_input.size(1),
            -1,
        )

        decoder_input = torch.cat(
            [embedded, repeated_context],
            dim=-1,
        )

        output, hidden = self.rnn(decoder_input, hidden)
        logits = self.output_layer(output)

        return logits, hidden


class Seq2Seq(nn.Module):
    """Combine the encoder and the context-fed decoder."""

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, source_lengths, target_input):
        context = self.encoder(source, source_lengths)
        hidden = context

        logits, _ = self.decoder(
            target_input,
            hidden,
            context,
        )

        return logits


encoder = Encoder(
    vocabulary_size=len(english_token_to_index),
    embedding_dimension=EMBEDDING_DIMENSION,
    hidden_dimension=HIDDEN_DIMENSION,
)

decoder = Decoder(
    vocabulary_size=len(italian_token_to_index),
    embedding_dimension=EMBEDDING_DIMENSION,
    hidden_dimension=HIDDEN_DIMENSION,
)

model = Seq2Seq(encoder, decoder).to(DEVICE)


# Check the model output dimensions using one small batch.
example_source = source_batch[:4].to(DEVICE)
example_lengths = source_lengths[:4]
example_target_input = target_batch[:4, :-1].to(DEVICE)

with torch.no_grad():
    example_logits = model(
        example_source,
        example_lengths,
        example_target_input,
    )

expected_shape = (
    example_target_input.size(0),
    example_target_input.size(1),
    len(italian_token_to_index),
)

assert example_logits.shape == expected_shape

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(model)
print(f"\nOutput shape: {example_logits.shape}")
print(f"Trainable parameters: {number_of_parameters:,}")

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(10000, 64, padding_idx=0)
    (rnn): RNN(64, 128, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(10000, 64, padding_idx=0)
    (rnn): RNN(192, 128, batch_first=True)
    (output_layer): Linear(in_features=128, out_features=10000, bias=True)
  )
)

Output shape: torch.Size([4, 21, 10000])
Trainable parameters: 2,636,048


## 4. Training and evaluation strategy

The model is trained using **full teacher forcing**. The target sentence is shifted by one position:

- the decoder input starts with `<sos>` and excludes the final token;
- the expected output excludes `<sos>` and ends with `<eos>`.

This allows the complete target sequence to be processed in one vectorized RNN operation, avoiding a slower Python loop over individual words during training.

The training strategy uses:

- **cross-entropy loss**, ignoring `<pad>` tokens;
- **Adam optimizer**, initially with a learning rate of 0.001;
- **gradient clipping**, which limits unstable gradients that can occur in vanilla RNNs;
- a maximum of **two epochs** for the final experiments;
- validation loss to select the best checkpoint;
- execution-time measurement for every experiment.

Early stopping is not used because the models are trained for very few epochs. Instead, the model state with the lowest validation loss is saved and restored after training.

BLEU and METEOR are calculated on translations generated with greedy decoding. BLEU is computed over the complete evaluated corpus, following the course recommendation, while METEOR is averaged over the individual sentences. Both scores are reported on a 0–100 scale, where higher values are better.

Language-specific stemming is used for METEOR. Synonym matching is disabled because equivalent multilingual lexical resources are not available for both English and Italian; this avoids evaluating the two translation directions differently.

The test set remains untouched during model development and will only be used after selecting the architecture and hyperparameters.

In [49]:
MAX_GRADIENT_NORM = 1.0


def train_one_epoch(model, data_loader, optimizer, loss_function):
    """Train the model for one epoch and return the mean token loss."""
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for source, source_lengths, target in data_loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)

        target_input = target[:, :-1]
        target_output = target[:, 1:]

        optimizer.zero_grad()

        logits = model(source, source_lengths, target_input)

        loss = loss_function(
            logits.reshape(-1, logits.size(-1)),
            target_output.reshape(-1),
        )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRADIENT_NORM,
        )

        optimizer.step()

        number_of_tokens = (
            target_output != PAD_INDEX
        ).sum().item()

        total_loss += loss.item() * number_of_tokens
        total_tokens += number_of_tokens

    return total_loss / total_tokens


@torch.no_grad()
def validate_one_epoch(model, data_loader, loss_function):
    """Evaluate the model and return the mean token loss."""
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for source, source_lengths, target in data_loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)

        target_input = target[:, :-1]
        target_output = target[:, 1:]

        logits = model(source, source_lengths, target_input)

        loss = loss_function(
            logits.reshape(-1, logits.size(-1)),
            target_output.reshape(-1),
        )

        number_of_tokens = (
            target_output != PAD_INDEX
        ).sum().item()

        total_loss += loss.item() * number_of_tokens
        total_tokens += number_of_tokens

    return total_loss / total_tokens

In [51]:
from matplotlib.style import context


try:
    from nltk.stem import SnowballStemmer
    from nltk.translate.bleu_score import corpus_bleu
    from nltk.translate.meteor_score import meteor_score
except ImportError as error:
    raise ImportError(
        "NLTK is required. Install it with: pip install nltk"
    ) from error


MAX_DECODE_LENGTH = MAX_SENTENCE_LENGTH + 1


class NoSynonyms:
    """Disable WordNet synonym matching in METEOR."""

    @staticmethod
    def synsets(word):
        return []


def indices_to_tokens(indices, index_to_token):
    """Convert generated indices into tokens."""
    tokens = []

    for index in indices:
        index = int(index)

        if index == EOS_INDEX:
            break

        if index not in (PAD_INDEX, SOS_INDEX):
            tokens.append(index_to_token[index])

    return tokens


@torch.no_grad()
def generate_translations(model, data_loader, target_index_to_token):
    """Generate translations for a complete DataLoader."""
    model.eval()

    hypotheses = []

    for source, source_lengths, _ in data_loader:
        source = source.to(DEVICE)

        context = model.encoder(source, source_lengths)
        hidden = context

        current_token = torch.full(
            (source.size(0), 1),
            SOS_INDEX,
            dtype=torch.long,
            device=DEVICE,
        )

        generated_tokens = []
        finished = torch.zeros(
            source.size(0),
            dtype=torch.bool,
            device=DEVICE,
        )

        for _ in range(MAX_DECODE_LENGTH):
            logits, hidden = model.decoder(
                current_token,
                hidden,
                context,
            )
            step_logits = logits[:, -1].clone()

            step_logits[
                :,
                [PAD_INDEX, UNK_INDEX, SOS_INDEX],
            ] = float("-inf")

            next_token = step_logits.argmax(dim=-1)

            next_token = torch.where(
                finished,
                torch.full_like(next_token, EOS_INDEX),
                next_token,
            )

            generated_tokens.append(next_token.cpu())
            finished |= next_token.eq(EOS_INDEX)
            current_token = next_token.unsqueeze(1)

            if finished.all():
                break

        generated_tokens = torch.stack(
            generated_tokens,
            dim=1,
        )

        hypotheses.extend(
            indices_to_tokens(sequence, target_index_to_token)
            for sequence in generated_tokens
        )

    references = [
        tokenize(target_sentence)
        for _, target_sentence in data_loader.dataset.pairs
    ]

    assert len(references) == len(hypotheses)

    return references, hypotheses


def compute_translation_metrics(
    references,
    hypotheses,
    target_language,
):
    """Compute corpus BLEU and mean sentence METEOR."""
    bleu = 100 * corpus_bleu(
        [[reference] for reference in references],
        hypotheses,
    )

    stemmer = SnowballStemmer(target_language)
    no_synonyms = NoSynonyms()

    meteor_values = [
        meteor_score(
            [reference],
            hypothesis,
            stemmer=stemmer,
            wordnet=no_synonyms,
        )
        for reference, hypothesis in zip(references, hypotheses)
    ]

    meteor = 100 * sum(meteor_values) / len(meteor_values)

    return {
        "BLEU": bleu,
        "METEOR": meteor,
    }

In [37]:
import time


def run_experiment(
    model,
    training_loader,
    validation_loader,
    checkpoint_name,
    number_of_epochs=2,
    learning_rate=0.001,
):
    """Train a model and restore its best validation checkpoint."""
    loss_function = nn.CrossEntropyLoss(
        ignore_index=PAD_INDEX,
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    checkpoint_path = MODEL_DIR / checkpoint_name
    best_validation_loss = float("inf")

    history = {
        "training_loss": [],
        "validation_loss": [],
    }

    start_time = time.perf_counter()

    for epoch in range(1, number_of_epochs + 1):
        training_loss = train_one_epoch(
            model,
            training_loader,
            optimizer,
            loss_function,
        )
        validation_loss = validate_one_epoch(
            model,
            validation_loader,
            loss_function,
        )

        history["training_loss"].append(training_loss)
        history["validation_loss"].append(validation_loss)

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            torch.save(
                model.state_dict(),
                checkpoint_path,
            )

        print(
            f"Epoch {epoch}/{number_of_epochs} | "
            f"training loss: {training_loss:.4f} | "
            f"validation loss: {validation_loss:.4f}"
        )

    elapsed_seconds = time.perf_counter() - start_time

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=DEVICE,
        )
    )

    history["elapsed_seconds"] = elapsed_seconds
    history["best_validation_loss"] = best_validation_loss

    print(f"Best validation loss: {best_validation_loss:.4f}")
    print(f"Training time: {elapsed_seconds / 60:.2f} minutes")
    print(f"Checkpoint: {checkpoint_path}")

    return history

## 5. Manual hyperparameter tuning

The course material states that the validation set should be used for hyperparameter selection, while the test set should remain untouched until the final evaluation.

A small **manual tuning** procedure is used instead of an exhaustive grid search. This choice is appropriate because all experiments must run on CPU and the project requires several additional models.

Three configurations are compared:

| Configuration | Embedding dimension | Hidden dimension | Learning rate |
|---|---:|---:|---:|
| Small | 64 | 64 | 0.001 |
| Medium | 64 | 128 | 0.001 |
| Lower learning rate | 64 | 128 | 0.0005 |

The comparison changes one factor at a time:

1. `Small` and `Medium` measure the effect of increasing the RNN hidden-state capacity.
2. `Medium` and `Lower learning rate` measure the effect of a smaller optimization step.

The remaining choices are fixed to ensure a fair comparison:

- one vanilla RNN layer;
- vocabulary size of 5,000 tokens per language;
- maximum sentence length of 20 tokens;
- batch size of 128;
- Adam optimizer;
- two training epochs;
- gradient clipping with a maximum norm of 1.0.

Embedding size is not tuned here because a separate experiment will later compare different embedding approaches.

To reduce computation, every configuration is trained using the same 10,000 training pairs and evaluated using the same 2,000 validation pairs. Each model starts from a reproducible random initialization and sees the same training data order.

The main selection criterion is the lowest validation loss. Validation BLEU and METEOR are also reported to confirm whether the lower loss corresponds to better generated translations. If two configurations perform similarly, the smaller and faster model is preferred.

After selection, the best configuration will be trained again using the complete filtered training set and validated on the complete validation set. The test set is not used during tuning.

In [38]:
TUNING_TRAINING_SIZE = 10_000
TUNING_VALIDATION_SIZE = 2_000
TUNING_EPOCHS = 2


tuning_training_pairs = training_pairs[:TUNING_TRAINING_SIZE]
tuning_validation_pairs = validation_pairs[:TUNING_VALIDATION_SIZE]

tuning_training_dataset = TranslationDataset(
    tuning_training_pairs,
    english_token_to_index,
    italian_token_to_index,
)
tuning_validation_dataset = TranslationDataset(
    tuning_validation_pairs,
    english_token_to_index,
    italian_token_to_index,
)

tuning_validation_loader = DataLoader(
    tuning_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)


def create_model(
    source_vocabulary_size,
    target_vocabulary_size,
    embedding_dimension,
    hidden_dimension,
):
    """Create a new RNN encoder-decoder model."""
    encoder = Encoder(
        source_vocabulary_size,
        embedding_dimension,
        hidden_dimension,
    )
    decoder = Decoder(
        target_vocabulary_size,
        embedding_dimension,
        hidden_dimension,
    )

    return Seq2Seq(encoder, decoder).to(DEVICE)


tuning_configurations = [
    {
        "name": "small",
        "embedding_dimension": 64,
        "hidden_dimension": 64,
        "learning_rate": 0.001,
    },
    {
        "name": "medium",
        "embedding_dimension": 64,
        "hidden_dimension": 128,
        "learning_rate": 0.001,
    },
    {
        "name": "lower_learning_rate",
        "embedding_dimension": 64,
        "hidden_dimension": 128,
        "learning_rate": 0.0005,
    },
]

tuning_results = []

for configuration in tuning_configurations:
    print(f"\nConfiguration: {configuration['name']}")

    # Use the same initialization and training order for each run.
    torch.manual_seed(SEED)

    tuning_training_loader = DataLoader(
        tuning_training_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_batch,
        generator=torch.Generator().manual_seed(SEED),
    )

    tuning_model = create_model(
        source_vocabulary_size=len(english_token_to_index),
        target_vocabulary_size=len(italian_token_to_index),
        embedding_dimension=configuration["embedding_dimension"],
        hidden_dimension=configuration["hidden_dimension"],
    )

    history = run_experiment(
        model=tuning_model,
        training_loader=tuning_training_loader,
        validation_loader=tuning_validation_loader,
        checkpoint_name=f"tuning_{configuration['name']}.pt",
        number_of_epochs=TUNING_EPOCHS,
        learning_rate=configuration["learning_rate"],
    )

    references, hypotheses = generate_translations(
        tuning_model,
        tuning_validation_loader,
        italian_index_to_token,
    )
    metrics = compute_translation_metrics(
        references,
        hypotheses,
        target_language="italian",
    )

    best_epoch_index = int(
        np.argmin(history["validation_loss"])
    )

    number_of_parameters = sum(
        parameter.numel()
        for parameter in tuning_model.parameters()
    )

    tuning_results.append(
        {
            **configuration,
            "best_epoch": best_epoch_index + 1,
            "training_loss": history["training_loss"][best_epoch_index],
            "validation_loss": history["validation_loss"][best_epoch_index],
            "BLEU": metrics["BLEU"],
            "METEOR": metrics["METEOR"],
            "parameters": number_of_parameters,
            "training_minutes": history["elapsed_seconds"] / 60,
        }
    )

    print(f"Validation BLEU: {metrics['BLEU']:.2f}")
    print(f"Validation METEOR: {metrics['METEOR']:.2f}")


Configuration: small
Epoch 1/2 | training loss: 7.5355 | validation loss: 6.3269
Epoch 2/2 | training loss: 6.3382 | validation loss: 6.2150
Best validation loss: 6.2150
Training time: 0.63 minutes
Checkpoint: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\models\task3\tuning_small.pt
Validation BLEU: 0.00
Validation METEOR: 0.00

Configuration: medium
Epoch 1/2 | training loss: 7.0157 | validation loss: 6.2134
Epoch 2/2 | training loss: 6.1908 | validation loss: 6.0250
Best validation loss: 6.0250
Training time: 0.73 minutes
Checkpoint: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\models\task3\tuning_medium.pt


c:\Users\tella\AppData\Local\Programs\Python\Python310\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\tella\AppData\Local\Programs\Python\Python310\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\tella\AppData\Local\Programs\Python\Python310\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lowe

Validation BLEU: 0.00
Validation METEOR: 1.20

Configuration: lower_learning_rate
Epoch 1/2 | training loss: 7.5101 | validation loss: 6.2759
Epoch 2/2 | training loss: 6.3042 | validation loss: 6.1884
Best validation loss: 6.1884
Training time: 0.81 minutes
Checkpoint: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\models\task3\tuning_lower_learning_rate.pt
Validation BLEU: 0.00
Validation METEOR: 1.20


In [39]:
import pandas as pd
from IPython.display import display


tuning_results_table = pd.DataFrame(tuning_results)

tuning_results_table = tuning_results_table[
    [
        "name",
        "embedding_dimension",
        "hidden_dimension",
        "learning_rate",
        "best_epoch",
        "training_loss",
        "validation_loss",
        "BLEU",
        "METEOR",
        "parameters",
        "training_minutes",
    ]
].sort_values("validation_loss")

display(
    tuning_results_table.round(
        {
            "training_loss": 4,
            "validation_loss": 4,
            "BLEU": 2,
            "METEOR": 2,
            "training_minutes": 2,
        }
    )
)


best_configuration = min(
    tuning_results,
    key=lambda result: result["validation_loss"],
)

BEST_EMBEDDING_DIMENSION = best_configuration[
    "embedding_dimension"
]
BEST_HIDDEN_DIMENSION = best_configuration[
    "hidden_dimension"
]
BEST_LEARNING_RATE = best_configuration[
    "learning_rate"
]
#BEST_NUMBER_OF_EPOCHS = TUNING_EPOCHS
FINAL_NUMBER_OF_EPOCHS = 5

tuning_results_path = (
    RESULTS_DIR / "hyperparameter_tuning_results.csv"
)
tuning_results_table.to_csv(
    tuning_results_path,
    index=False,
)

print(f"Selected configuration: {best_configuration['name']}")
print(f"Embedding dimension: {BEST_EMBEDDING_DIMENSION}")
print(f"Hidden dimension: {BEST_HIDDEN_DIMENSION}")
print(f"Learning rate: {BEST_LEARNING_RATE}")
print(f"Saved results: {tuning_results_path}")

,name,embedding_dimension,hidden_dimension,learning_rate,best_epoch,training_loss,validation_loss,BLEU,METEOR,parameters,training_minutes
1,medium,64,128,0.0010,2,6.1908,6.0250,0.0,1.2,2619664,0.73
2,lower_learning_rate,64,128,0.0005,2,6.3042,6.1884,0.0,1.2,2619664,0.81
0,small,64,64,0.0010,2,6.3382,6.2150,0.0,0.0,1946640,0.63


Selected configuration: medium
Embedding dimension: 64
Hidden dimension: 128
Learning rate: 0.001
Saved results: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\results\task3_outputs\hyperparameter_tuning_results.csv


In [ ]:
#sanity check - to be deleted

from collections import Counter


diagnostic_model = create_model(
    source_vocabulary_size=len(english_token_to_index),
    target_vocabulary_size=len(italian_token_to_index),
    embedding_dimension=64,
    hidden_dimension=128,
)

diagnostic_model.load_state_dict(
    torch.load(
        MODEL_DIR / "tuning_medium.pt",
        map_location=DEVICE,
    )
)

diagnostic_references, diagnostic_hypotheses = generate_translations(
    diagnostic_model,
    tuning_validation_loader,
    italian_index_to_token,
)

generated_lengths = [
    len(hypothesis)
    for hypothesis in diagnostic_hypotheses
]

empty_percentage = (
    100
    * sum(length == 0 for length in generated_lengths)
    / len(generated_lengths)
)

print(f"Empty translations: {empty_percentage:.1f}%")
print(f"Mean generated length: {np.mean(generated_lengths):.2f} tokens")
print(
    f"Unique translations: "
    f"{len(set(map(tuple, diagnostic_hypotheses))):,}"
)

print("\nMost common generated translations:")
for hypothesis, count in Counter(
    map(tuple, diagnostic_hypotheses)
).most_common(5):
    print(f"{count:4d} × {' '.join(hypothesis) or '<empty>'}")

print("\nTranslation examples:")
for i in range(10):
    print(f"\nEN:  {tuning_validation_pairs[i][0]}")
    print(f"REF: {' '.join(diagnostic_references[i])}")
    print(f"HYP: {' '.join(diagnostic_hypotheses[i]) or '<empty>'}")

Empty translations: 0.0%
Mean generated length: 2.00 tokens
Unique translations: 1

Most common generated translations:
2000 × la <unk>

Translation examples:

EN:  ... which is a fundamental element in the fight against organised crime and its links with terrorism?
REF: ...elemento fondamentale per la lotta al crimine organizzato e ai suoi legami con il terrorismo?
HYP: la <unk>

EN:  "my group was appalled, politically speaking, at the blandness of the terms of the socialist pseudo-censure motion of december."
REF: "il mio gruppo, politicamente parlando, si è stupito della mitezza del testo della pseudomozione di censura socialista presentata in dicembre."
HYP: la <unk>

EN:  "ultimately, the decisions were taken at the highest political level possible."
REF: "infine, le decisioni sono state prese al più alto livello politico possibile."
HYP: la <unk>

EN:  "of course, saddam hussein must be made to change, but his people must change too."
REF: "sicuramente saddam deve essere cambiat

## 6. Final English-to-Italian word model

The manual tuning selected the `medium` configuration:

- embedding dimension: 64;
- hidden dimension: 128;
- learning rate: 0.001;
- one-layer vanilla RNN;
- vocabulary size: 10,000 tokens per language.

This configuration achieved the lowest validation loss with only a small increase in training time compared with the smaller model.

The tuning models were trained for only two epochs on 10,000 sentence pairs. This was sufficient for comparing validation losses but not for learning meaningful translations: the selected model generated the same short output for every sentence.

The final model is therefore trained from a new random initialization using:

- the complete filtered training set;
- the complete validation set;
- a maximum of five epochs;
- the checkpoint with the lowest validation loss.

Five epochs correspond to substantially more parameter updates than the short tuning runs, while the measured training speed remains compatible with CPU execution.

Before evaluating the test set, the final model is checked on the complete validation set. Translation diversity, generated length, BLEU, METEOR, and qualitative examples are inspected to verify that the model has not collapsed. The test set remains untouched until this validation check is completed.

In [52]:
FINAL_NUMBER_OF_EPOCHS = 20


# Recreate the training loader with a fresh deterministic shuffle.
final_training_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(SEED),
)

# Train a new model using the selected hyperparameters.
torch.manual_seed(SEED)

english_to_italian_model = create_model(
    source_vocabulary_size=len(english_token_to_index),
    target_vocabulary_size=len(italian_token_to_index),
    embedding_dimension=BEST_EMBEDDING_DIMENSION,
    hidden_dimension=BEST_HIDDEN_DIMENSION,
)

english_to_italian_history = run_experiment(
    model=english_to_italian_model,
    training_loader=final_training_loader,
    validation_loader=validation_loader,
    checkpoint_name="english_to_italian_word_rnn.pt",
    number_of_epochs=FINAL_NUMBER_OF_EPOCHS,
    learning_rate=BEST_LEARNING_RATE,
)

Epoch 1/20 | training loss: 6.2127 | validation loss: 5.6078
Epoch 2/20 | training loss: 5.4361 | validation loss: 5.1196
Epoch 3/20 | training loss: 5.0232 | validation loss: 4.8415
Epoch 4/20 | training loss: 4.7479 | validation loss: 4.6498
Epoch 5/20 | training loss: 4.5490 | validation loss: 4.5260
Epoch 6/20 | training loss: 4.3973 | validation loss: 4.4426
Epoch 7/20 | training loss: 4.2796 | validation loss: 4.3850
Epoch 8/20 | training loss: 4.1815 | validation loss: 4.3390
Epoch 9/20 | training loss: 4.0970 | validation loss: 4.2989
Epoch 10/20 | training loss: 4.0224 | validation loss: 4.2710
Epoch 11/20 | training loss: 3.9556 | validation loss: 4.2549
Epoch 12/20 | training loss: 3.8950 | validation loss: 4.2367
Epoch 13/20 | training loss: 3.8400 | validation loss: 4.2274
Epoch 14/20 | training loss: 3.7903 | validation loss: 4.2142
Epoch 15/20 | training loss: 3.7436 | validation loss: 4.2097
Epoch 16/20 | training loss: 3.6991 | validation loss: 4.2086
Epoch 17/20 | tra

In [53]:
validation_references, validation_hypotheses = generate_translations(
    english_to_italian_model,
    validation_loader,
    italian_index_to_token,
)

english_to_italian_validation_metrics = compute_translation_metrics(
    validation_references,
    validation_hypotheses,
    target_language="italian",
)

generated_lengths = [
    len(hypothesis)
    for hypothesis in validation_hypotheses
]

empty_percentage = (
    100
    * sum(length == 0 for length in generated_lengths)
    / len(generated_lengths)
)

unique_translations = len(
    set(map(tuple, validation_hypotheses))
)

print("Validation results")
print(
    f"BLEU: "
    f"{english_to_italian_validation_metrics['BLEU']:.2f}"
)
print(
    f"METEOR: "
    f"{english_to_italian_validation_metrics['METEOR']:.2f}"
)
print(f"Empty translations: {empty_percentage:.1f}%")
print(
    f"Mean generated length: "
    f"{np.mean(generated_lengths):.2f} tokens"
)
print(f"Unique translations: {unique_translations:,}")

print("\nMost common generated translations:")
for hypothesis, count in Counter(
    map(tuple, validation_hypotheses)
).most_common(5):
    generated_text = " ".join(hypothesis) or "<empty>"
    print(f"{count:4d} × {generated_text}")

print("\nTranslation examples:")
for i in range(5):
    print(f"\nEN:  {validation_pairs[i][0]}")
    print(f"REF: {' '.join(validation_references[i])}")
    print(
        f"HYP: "
        f"{' '.join(validation_hypotheses[i]) or '<empty>'}"
    )

Validation results
BLEU: 1.84
METEOR: 11.11
Empty translations: 0.0%
Mean generated length: 14.25 tokens
Unique translations: 6,245

Most common generated translations:
  51 × la discussione è chiusa.
  31 × (applausi)
  25 × la votazione si svolgerà alle 12.00.
  25 × .
  25 × questo è il problema.

Translation examples:

EN:  ... which is a fundamental element in the fight against organised crime and its links with terrorism?
REF: ...elemento fondamentale per la lotta al crimine organizzato e ai suoi legami con il terrorismo?
HYP: "in secondo luogo, la commissione europea e la commissione ha bisogno di un accordo tra le autorità di sicurezza e di

EN:  "my group was appalled, politically speaking, at the blandness of the terms of the socialist pseudo-censure motion of december."
REF: "il mio gruppo, politicamente parlando, si è stupito della mitezza del testo della pseudomozione di censura socialista presentata in dicembre."
HYP: "la commissione ha proposto un accordo con il relatore

Validation results
BLEU: 1.84
METEOR: 11.11
Empty translations: 0.0%
Mean generated length: 14.25 tokens
Unique translations: 6,245

Most common generated translations:
  51 × la discussione è chiusa.
  31 × (applausi)
  25 × la votazione si svolgerà alle 12.00.
  25 × .
  25 × questo è il problema.

Translation examples:

EN:  ... which is a fundamental element in the fight against organised crime and its links with terrorism?
REF: ...elemento fondamentale per la lotta al crimine organizzato e ai suoi legami con il terrorismo?
HYP: "in secondo luogo, la commissione europea e la commissione ha bisogno di un accordo tra le autorità di sicurezza e di

EN:  "my group was appalled, politically speaking, at the blandness of the terms of the socialist pseudo-censure motion of december."
REF: "il mio gruppo, politicamente parlando, si è stupito della mitezza del testo della pseudomozione di censura socialista presentata in dicembre."
HYP: "la commissione ha proposto un accordo con il relatore e la commissione ha appena detto."

EN:  "ultimately, the decisions were taken at the highest political level possible."
...

EN:  "i can hear the voice of your colleague now, announcing that we will certainly reach a compromise on 29 september."
REF: sono lieto di sentire dal suo collega l'annuncio che già entro il 29 settembre verrà raggiunto un compromesso.
HYP: "per quanto riguarda la relazione dell'onorevole parlamentare ha fatto riferimento alla commissione per il suo gruppo di vista della commissione non


In [61]:
class LSTMEncoder(nn.Module):
    """Encode the English sentence with a single-layer LSTM."""

    def __init__(
        self,
        vocabulary_size,
        embedding_dimension,
        hidden_dimension,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dimension,
            padding_idx=PAD_INDEX,
        )

        self.lstm = nn.LSTM(
            embedding_dimension,
            hidden_dimension,
            batch_first=True,
        )

    def forward(self, source, source_lengths):
        embedded = self.embedding(source)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, state = self.lstm(packed)

        return state


class LSTMDecoder(nn.Module):
    """Decode using the target prefix and fixed encoder context."""

    def __init__(
        self,
        vocabulary_size,
        embedding_dimension,
        hidden_dimension,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dimension,
            padding_idx=PAD_INDEX,
        )

        self.lstm = nn.LSTM(
            embedding_dimension + hidden_dimension,
            hidden_dimension,
            batch_first=True,
        )

        self.output_layer = nn.Linear(
            hidden_dimension,
            vocabulary_size,
        )

    def forward(self, target_input, state, context):
        embedded = self.embedding(target_input)

        encoder_hidden = context[0][-1]

        repeated_context = encoder_hidden.unsqueeze(1).expand(
            -1,
            target_input.size(1),
            -1,
        )

        decoder_input = torch.cat(
            [embedded, repeated_context],
            dim=-1,
        )

        output, state = self.lstm(decoder_input, state)
        logits = self.output_layer(output)

        return logits, state


class LSTMSeq2Seq(nn.Module):
    """Combine the context-fed LSTM encoder and decoder."""

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, source_lengths, target_input):
        context = self.encoder(source, source_lengths)
        decoder_state = context

        logits, _ = self.decoder(
            target_input,
            decoder_state,
            context,
        )

        return logits


def create_lstm_model(
    source_vocabulary_size,
    target_vocabulary_size,
    embedding_dimension,
    hidden_dimension,
):
    """Create a context-fed LSTM encoder-decoder model."""

    encoder = LSTMEncoder(
        vocabulary_size=source_vocabulary_size,
        embedding_dimension=embedding_dimension,
        hidden_dimension=hidden_dimension,
    )

    decoder = LSTMDecoder(
        vocabulary_size=target_vocabulary_size,
        embedding_dimension=embedding_dimension,
        hidden_dimension=hidden_dimension,
    )

    return LSTMSeq2Seq(encoder, decoder).to(DEVICE)

In [62]:
LSTM_EMBEDDING_DIMENSION = 64
LSTM_HIDDEN_DIMENSION = 128
LSTM_LEARNING_RATE = 0.001
LSTM_NUMBER_OF_EPOCHS = 20

lstm_model = create_lstm_model(
    source_vocabulary_size=len(english_token_to_index),
    target_vocabulary_size=len(italian_token_to_index),
    embedding_dimension=LSTM_EMBEDDING_DIMENSION,
    hidden_dimension=LSTM_HIDDEN_DIMENSION,
)

source_batch, source_lengths, target_batch = next(
    iter(training_loader)
)

source_batch = source_batch.to(DEVICE)
target_batch = target_batch.to(DEVICE)

with torch.no_grad():
    test_logits = lstm_model(
        source_batch,
        source_lengths,
        target_batch[:, :-1],
    )

expected_shape = (
    source_batch.size(0),
    target_batch.size(1) - 1,
    len(italian_token_to_index),
)

assert test_logits.shape == expected_shape

number_of_parameters = sum(
    parameter.numel()
    for parameter in lstm_model.parameters()
)

print(f"Source batch: {source_batch.shape}")
print(f"Target input: {target_batch[:, :-1].shape}")
print(f"Model output: {test_logits.shape}")
print(f"Trainable parameters: {number_of_parameters:,}")

Source batch: torch.Size([128, 22])
Target input: torch.Size([128, 21])
Model output: torch.Size([128, 21, 10000])
Trainable parameters: 2,834,192


In [63]:
# Recreate the training loader with the same deterministic shuffle
# used for the context-fed RNN experiment.
lstm_training_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(SEED),
)

# Use the same initialization seed for a fair comparison.
torch.manual_seed(SEED)

lstm_model = create_lstm_model(
    source_vocabulary_size=len(english_token_to_index),
    target_vocabulary_size=len(italian_token_to_index),
    embedding_dimension=LSTM_EMBEDDING_DIMENSION,
    hidden_dimension=LSTM_HIDDEN_DIMENSION,
)

lstm_history = run_experiment(
    model=lstm_model,
    training_loader=lstm_training_loader,
    validation_loader=validation_loader,
    checkpoint_name="english_to_italian_context_lstm.pt",
    number_of_epochs=LSTM_NUMBER_OF_EPOCHS,
    learning_rate=LSTM_LEARNING_RATE,
)

Epoch 1/20 | training loss: 6.2673 | validation loss: 5.7256
Epoch 2/20 | training loss: 5.5506 | validation loss: 5.2080
Epoch 3/20 | training loss: 5.0872 | validation loss: 4.8531
Epoch 4/20 | training loss: 4.7674 | validation loss: 4.6229
Epoch 5/20 | training loss: 4.5384 | validation loss: 4.4656
Epoch 6/20 | training loss: 4.3606 | validation loss: 4.3397
Epoch 7/20 | training loss: 4.2157 | validation loss: 4.2462
Epoch 8/20 | training loss: 4.0939 | validation loss: 4.1739
Epoch 9/20 | training loss: 3.9880 | validation loss: 4.1126
Epoch 10/20 | training loss: 3.8939 | validation loss: 4.0662
Epoch 11/20 | training loss: 3.8092 | validation loss: 4.0251
Epoch 12/20 | training loss: 3.7316 | validation loss: 3.9946
Epoch 13/20 | training loss: 3.6605 | validation loss: 3.9655
Epoch 14/20 | training loss: 3.5934 | validation loss: 3.9455
Epoch 15/20 | training loss: 3.5315 | validation loss: 3.9280
Epoch 16/20 | training loss: 3.4728 | validation loss: 3.9124
Epoch 17/20 | tra

In [64]:
lstm_validation_references, lstm_validation_hypotheses = (
    generate_translations(
        lstm_model,
        validation_loader,
        italian_index_to_token,
    )
)

lstm_validation_metrics = compute_translation_metrics(
    lstm_validation_references,
    lstm_validation_hypotheses,
    target_language="italian",
)

generated_lengths = [
    len(hypothesis)
    for hypothesis in lstm_validation_hypotheses
]

empty_percentage = (
    100
    * sum(length == 0 for length in generated_lengths)
    / len(generated_lengths)
)

unique_translations = len(
    set(map(tuple, lstm_validation_hypotheses))
)

print("LSTM validation results")
print(f"BLEU: {lstm_validation_metrics['BLEU']:.2f}")
print(f"METEOR: {lstm_validation_metrics['METEOR']:.2f}")
print(f"Empty translations: {empty_percentage:.1f}%")
print(
    f"Mean generated length: "
    f"{np.mean(generated_lengths):.2f} tokens"
)
print(f"Unique translations: {unique_translations:,}")

print("\nMost common generated translations:")
for hypothesis, count in Counter(
    map(tuple, lstm_validation_hypotheses)
).most_common(5):
    generated_text = " ".join(hypothesis) or "<empty>"
    print(f"{count:4d} × {generated_text}")

print("\nTranslation examples:")
for i in range(5):
    print(f"\nEN:  {validation_pairs[i][0]}")
    print(
        f"REF: "
        f"{' '.join(lstm_validation_references[i])}"
    )
    print(
        f"HYP: "
        f"{' '.join(lstm_validation_hypotheses[i]) or '<empty>'}"
    )

LSTM validation results
BLEU: 3.25
METEOR: 16.53
Empty translations: 0.0%
Mean generated length: 14.59 tokens
Unique translations: 6,790

Most common generated translations:
  39 × la discussione è chiusa.
  32 × (applausi)
  26 × .
  14 × questo è il problema.
  12 × "la votazione si svolgerà domani, alle 12.00."

Translation examples:

EN:  ... which is a fundamental element in the fight against organised crime and its links with terrorism?
REF: ...elemento fondamentale per la lotta al crimine organizzato e ai suoi legami con il terrorismo?
HYP: il problema è un ruolo di sviluppo sostenibile e la libertà di euro per la sicurezza alimentare.

EN:  "my group was appalled, politically speaking, at the blandness of the terms of the socialist pseudo-censure motion of december."
REF: "il mio gruppo, politicamente parlando, si è stupito della mitezza del testo della pseudomozione di censura socialista presentata in dicembre."
HYP: "il mio gruppo è stato detto che la commissione ha fatto rif

## LSTM-specific hyperparameter tuning

The previous experiments showed that the context-fed LSTM clearly outperformed the equivalent vanilla RNN. Since the earlier hyperparameter tuning was performed using the RNN architecture, a short additional tuning experiment is conducted for the selected LSTM architecture.

A manual tuning strategy is used to keep computation manageable on CPU. Four configurations are compared:

| Configuration | Embedding dimension | Hidden dimension | Learning rate |
|---|---:|---:|---:|
| Small | 64 | 64 | 0.001 |
| Medium | 64 | 128 | 0.001 |
| Large | 64 | 256 | 0.001 |
| Lower learning rate | 64 | 128 | 0.0005 |

The configurations test the effect of recurrent-state capacity and learning rate while keeping all other settings fixed. Every model is trained for five epochs using the same 10,000 training pairs and evaluated on the same 2,000 validation pairs.

The primary selection criterion is validation loss. BLEU and METEOR are also reported to check whether the validation-loss differences correspond to differences in generated translation quality. If two configurations perform similarly, the smaller model will be preferred because it requires less computation.

In [66]:
LSTM_TUNING_EPOCHS = 5

lstm_tuning_configurations = [
    {
        "name": "small",
        "embedding_dimension": 64,
        "hidden_dimension": 64,
        "learning_rate": 0.001,
    },
    {
        "name": "medium",
        "embedding_dimension": 64,
        "hidden_dimension": 128,
        "learning_rate": 0.001,
    },
    {
        "name": "large",
        "embedding_dimension": 64,
        "hidden_dimension": 256,
        "learning_rate": 0.001,
    },
    {
        "name": "lower_learning_rate",
        "embedding_dimension": 64,
        "hidden_dimension": 128,
        "learning_rate": 0.0005,
    },
]

lstm_tuning_results = []

for configuration in lstm_tuning_configurations:
    print(f"\nConfiguration: {configuration['name']}")

    # Use the same initialization and data order for every run.
    torch.manual_seed(SEED)

    lstm_tuning_training_loader = DataLoader(
        tuning_training_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_batch,
        generator=torch.Generator().manual_seed(SEED),
    )

    tuning_lstm_model = create_lstm_model(
        source_vocabulary_size=len(english_token_to_index),
        target_vocabulary_size=len(italian_token_to_index),
        embedding_dimension=configuration["embedding_dimension"],
        hidden_dimension=configuration["hidden_dimension"],
    )

    history = run_experiment(
        model=tuning_lstm_model,
        training_loader=lstm_tuning_training_loader,
        validation_loader=tuning_validation_loader,
        checkpoint_name=(
            f"lstm_tuning_{configuration['name']}.pt"
        ),
        number_of_epochs=LSTM_TUNING_EPOCHS,
        learning_rate=configuration["learning_rate"],
    )

    references, hypotheses = generate_translations(
        tuning_lstm_model,
        tuning_validation_loader,
        italian_index_to_token,
    )

    metrics = compute_translation_metrics(
        references,
        hypotheses,
        target_language="italian",
    )

    best_epoch_index = int(
        np.argmin(history["validation_loss"])
    )

    number_of_parameters = sum(
        parameter.numel()
        for parameter in tuning_lstm_model.parameters()
    )

    lstm_tuning_results.append(
        {
            **configuration,
            "best_epoch": best_epoch_index + 1,
            "training_loss": history[
                "training_loss"
            ][best_epoch_index],
            "validation_loss": history[
                "validation_loss"
            ][best_epoch_index],
            "BLEU": metrics["BLEU"],
            "METEOR": metrics["METEOR"],
            "parameters": number_of_parameters,
            "training_minutes": (
                history["elapsed_seconds"] / 60
            ),
        }
    )

    print(f"Validation BLEU: {metrics['BLEU']:.2f}")
    print(f"Validation METEOR: {metrics['METEOR']:.2f}")


Configuration: small
Epoch 1/5 | training loss: 7.4500 | validation loss: 6.3106
Epoch 2/5 | training loss: 6.3599 | validation loss: 6.2430
Epoch 3/5 | training loss: 6.2577 | validation loss: 6.1407
Epoch 4/5 | training loss: 6.1388 | validation loss: 6.0341
Epoch 5/5 | training loss: 6.0234 | validation loss: 5.9433
Best validation loss: 5.9433
Training time: 1.51 minutes
Checkpoint: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\models\task3\lstm_tuning_small.pt
Validation BLEU: 0.00
Validation METEOR: 2.76

Configuration: medium
Epoch 1/5 | training loss: 7.0350 | validation loss: 6.2173
Epoch 2/5 | training loss: 6.2214 | validation loss: 6.0958
Epoch 3/5 | training loss: 6.0952 | validation loss: 5.9896
Epoch 4/5 | training loss: 5.9610 | validation loss: 5.8707
Epoch 5/5 | training loss: 5.8194 | validation loss: 5.7708
Best validation loss: 5.7708
Training time: 2.00 minutes
Checkpoint: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_

In [67]:
lstm_tuning_results_table = pd.DataFrame(
    lstm_tuning_results
)

lstm_tuning_results_table = lstm_tuning_results_table[
    [
        "name",
        "embedding_dimension",
        "hidden_dimension",
        "learning_rate",
        "best_epoch",
        "training_loss",
        "validation_loss",
        "BLEU",
        "METEOR",
        "parameters",
        "training_minutes",
    ]
].sort_values("validation_loss")

display(
    lstm_tuning_results_table.round(
        {
            "training_loss": 4,
            "validation_loss": 4,
            "BLEU": 2,
            "METEOR": 2,
            "training_minutes": 2,
        }
    )
)

best_lstm_configuration = min(
    lstm_tuning_results,
    key=lambda result: result["validation_loss"],
)

BEST_LSTM_EMBEDDING_DIMENSION = (
    best_lstm_configuration["embedding_dimension"]
)
BEST_LSTM_HIDDEN_DIMENSION = (
    best_lstm_configuration["hidden_dimension"]
)
BEST_LSTM_LEARNING_RATE = (
    best_lstm_configuration["learning_rate"]
)

lstm_tuning_results_path = (
    RESULTS_DIR
    / "lstm_hyperparameter_tuning_results.csv"
)

lstm_tuning_results_table.to_csv(
    lstm_tuning_results_path,
    index=False,
)

print(
    f"Selected configuration: "
    f"{best_lstm_configuration['name']}"
)
print(
    f"Embedding dimension: "
    f"{BEST_LSTM_EMBEDDING_DIMENSION}"
)
print(
    f"Hidden dimension: "
    f"{BEST_LSTM_HIDDEN_DIMENSION}"
)
print(
    f"Learning rate: "
    f"{BEST_LSTM_LEARNING_RATE}"
)
print(f"Saved results: {lstm_tuning_results_path}")

,name,embedding_dimension,hidden_dimension,learning_rate,best_epoch,training_loss,validation_loss,BLEU,METEOR,parameters,training_minutes
2,large,64,256,0.0010,5,5.5152,5.5107,0.36,5.31,4771600,3.21
1,medium,64,128,0.0010,5,5.8194,5.7708,0.00,4.17,2834192,2.00
0,small,64,64,0.0010,5,6.0234,5.9433,0.00,2.76,2012944,1.51
3,lower_learning_rate,64,128,0.0005,5,6.0636,6.0029,0.00,3.72,2834192,1.98


Selected configuration: large
Embedding dimension: 64
Hidden dimension: 256
Learning rate: 0.001
Saved results: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\results\task3_outputs\lstm_hyperparameter_tuning_results.csv


## Final large LSTM experiment

The LSTM-specific manual tuning selected the large configuration with an embedding dimension of 64, a hidden dimension of 256 and a learning rate of 0.001.

This configuration achieved the lowest validation loss and the highest BLEU and METEOR scores under the same tuning budget. Although it contains more parameters than the medium model, its measured CPU training time remains manageable.

The selected configuration is now trained from a new random initialization using the complete training and validation sets. The best checkpoint is selected according to validation loss and compared with the previously trained medium LSTM.

In [69]:
LARGE_LSTM_NUMBER_OF_EPOCHS = 20

# Recreate the loader with a deterministic shuffle.
large_lstm_training_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(SEED),
)

# Reproduce the initialisation used in the previous experiments.
torch.manual_seed(SEED)

large_lstm_model = create_lstm_model(
    source_vocabulary_size=len(english_token_to_index),
    target_vocabulary_size=len(italian_token_to_index),
    embedding_dimension=BEST_LSTM_EMBEDDING_DIMENSION,
    hidden_dimension=BEST_LSTM_HIDDEN_DIMENSION,
)

large_lstm_history = run_experiment(
    model=large_lstm_model,
    training_loader=large_lstm_training_loader,
    validation_loader=validation_loader,
    checkpoint_name="english_to_italian_context_lstm_large.pt",
    number_of_epochs=LARGE_LSTM_NUMBER_OF_EPOCHS,
    learning_rate=BEST_LSTM_LEARNING_RATE,
)

Epoch 1/20 | training loss: 6.0634 | validation loss: 5.4337
Epoch 2/20 | training loss: 5.1926 | validation loss: 4.8417
Epoch 3/20 | training loss: 4.6892 | validation loss: 4.5106
Epoch 4/20 | training loss: 4.3638 | validation loss: 4.2939
Epoch 5/20 | training loss: 4.1241 | validation loss: 4.1518
Epoch 6/20 | training loss: 3.9326 | validation loss: 4.0455
Epoch 7/20 | training loss: 3.7705 | validation loss: 3.9680
Epoch 8/20 | training loss: 3.6277 | validation loss: 3.9041
Epoch 9/20 | training loss: 3.4990 | validation loss: 3.8561
Epoch 10/20 | training loss: 3.3783 | validation loss: 3.8234
Epoch 11/20 | training loss: 3.2684 | validation loss: 3.8003
Epoch 12/20 | training loss: 3.1640 | validation loss: 3.7889
Epoch 13/20 | training loss: 3.0644 | validation loss: 3.7817
Epoch 14/20 | training loss: 2.9695 | validation loss: 3.7823
Epoch 15/20 | training loss: 2.8789 | validation loss: 3.7816
Epoch 16/20 | training loss: 2.7926 | validation loss: 3.7904
Epoch 17/20 | tra

In [70]:
large_lstm_validation_references, large_lstm_validation_hypotheses = (
    generate_translations(
        large_lstm_model,
        validation_loader,
        italian_index_to_token,
    )
)

large_lstm_validation_metrics = compute_translation_metrics(
    large_lstm_validation_references,
    large_lstm_validation_hypotheses,
    target_language="italian",
)

generated_lengths = [
    len(hypothesis)
    for hypothesis in large_lstm_validation_hypotheses
]

empty_percentage = (
    100
    * sum(length == 0 for length in generated_lengths)
    / len(generated_lengths)
)

unique_translations = len(
    set(map(tuple, large_lstm_validation_hypotheses))
)

print("Large LSTM validation results")
print(
    f"BLEU: "
    f"{large_lstm_validation_metrics['BLEU']:.2f}"
)
print(
    f"METEOR: "
    f"{large_lstm_validation_metrics['METEOR']:.2f}"
)
print(f"Empty translations: {empty_percentage:.1f}%")
print(
    f"Mean generated length: "
    f"{np.mean(generated_lengths):.2f} tokens"
)
print(f"Unique translations: {unique_translations:,}")

print("\nMost common generated translations:")
for hypothesis, count in Counter(
    map(tuple, large_lstm_validation_hypotheses)
).most_common(5):
    generated_text = " ".join(hypothesis) or "<empty>"
    print(f"{count:4d} × {generated_text}")

print("\nTranslation examples:")
for i in range(5):
    print(f"\nEN:  {validation_pairs[i][0]}")
    print(
        f"REF: "
        f"{' '.join(large_lstm_validation_references[i])}"
    )
    print(
        f"HYP: "
        f"{' '.join(large_lstm_validation_hypotheses[i]) or '<empty>'}"
    )

Large LSTM validation results
BLEU: 3.92
METEOR: 18.23
Empty translations: 0.0%
Mean generated length: 14.22 tokens
Unique translations: 6,811

Most common generated translations:
  38 × la discussione è chiusa.
  31 × (applausi)
  13 × "la votazione si svolgerà domani, alle 12.00."
  12 × .
  11 × come?

Translation examples:

EN:  ... which is a fundamental element in the fight against organised crime and its links with terrorism?
REF: ...elemento fondamentale per la lotta al crimine organizzato e ai suoi legami con il terrorismo?
HYP: un altro esempio è la responsabilità di sicurezza alimentare e la lotta contro la povertà e la povertà."

EN:  "my group was appalled, politically speaking, at the blandness of the terms of the socialist pseudo-censure motion of december."
REF: "il mio gruppo, politicamente parlando, si è stupito della mitezza del testo della pseudomozione di censura socialista presentata in dicembre."
HYP: "il mio gruppo è stato accolto alla commissione, che è stato e

In [71]:
lstm_model_comparison = pd.DataFrame(
    [
        {
            "model": "medium_lstm",
            "embedding_dimension": 64,
            "hidden_dimension": 128,
            "validation_loss": lstm_history[
                "best_validation_loss"
            ],
            "BLEU": lstm_validation_metrics["BLEU"],
            "METEOR": lstm_validation_metrics["METEOR"],
            "training_minutes": (
                lstm_history["elapsed_seconds"] / 60
            ),
        },
        {
            "model": "large_lstm",
            "embedding_dimension": (
                BEST_LSTM_EMBEDDING_DIMENSION
            ),
            "hidden_dimension": (
                BEST_LSTM_HIDDEN_DIMENSION
            ),
            "validation_loss": large_lstm_history[
                "best_validation_loss"
            ],
            "BLEU": large_lstm_validation_metrics["BLEU"],
            "METEOR": large_lstm_validation_metrics["METEOR"],
            "training_minutes": (
                large_lstm_history["elapsed_seconds"] / 60
            ),
        },
    ]
)

display(lstm_model_comparison.round(2))

comparison_path = (
    RESULTS_DIR / "lstm_model_comparison.csv"
)

lstm_model_comparison.to_csv(
    comparison_path,
    index=False,
)

print(f"Saved comparison: {comparison_path}")

,model,embedding_dimension,hidden_dimension,validation_loss,BLEU,METEOR,training_minutes
0,medium_lstm,64,128,3.89,3.25,16.53,40.03
1,large_lstm,64,256,3.78,3.92,18.23,63.87


Saved comparison: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\results\task3_outputs\lstm_model_comparison.csv


## 7. Embedding comparison

The assignment requires tracking the impact of different embedding approaches on translation performance. Two alternatives are compared while keeping the selected Seq2Seq architecture and all training hyperparameters unchanged.

### Learned embeddings

The first approach corresponds to the current large LSTM model. Its English and Italian embedding matrices are randomly initialised and learned jointly with the translation model. This allows the embeddings to become directly specialised for the translation objective, but they initially contain no information about relationships between words.

### Word2Vec initialisation

The second approach uses Word2Vec, which is presented in the course material and implemented through Gensim. Two separate Word2Vec models are trained:

- an English model using only the English sentences in the training set;
- an Italian model using only the Italian sentences in the training set.

Using only the training split prevents information from the validation and test sets from affecting model development.

The Word2Vec models use the CBOW architecture, an embedding dimension of 64, a context window of 5 words and a minimum frequency of 2. CBOW is selected because it is simple and computationally efficient for the available corpus.

The resulting vectors initialise the encoder and decoder embedding layers. Words not represented by Word2Vec retain their random initialisation. The embedding layers remain trainable during Seq2Seq training so that the general distributional representations can adapt to the translation objective.

The Word2Vec-initialised model uses the same large LSTM architecture, training data, random seed, learning rate and epoch budget as the learned-embedding baseline. Validation loss is the primary comparison criterion, while BLEU and METEOR measure the impact on generated translations. The test set remains unused during this comparison.

In [72]:
try:
    from gensim.models import Word2Vec
except ImportError as error:
    raise ImportError(
        "Gensim is required. Install it with: pip install gensim"
    ) from error


WORD2VEC_DIMENSION = BEST_LSTM_EMBEDDING_DIMENSION
WORD2VEC_WINDOW = 5
WORD2VEC_MIN_COUNT = 2
WORD2VEC_EPOCHS = 10


english_word2vec_sentences = [
    tokenize(english)
    for english, _ in training_pairs
]

italian_word2vec_sentences = [
    tokenize(italian)
    for _, italian in training_pairs
]


word2vec_start_time = time.perf_counter()

english_word2vec_model = Word2Vec(
    sentences=english_word2vec_sentences,
    vector_size=WORD2VEC_DIMENSION,
    window=WORD2VEC_WINDOW,
    min_count=WORD2VEC_MIN_COUNT,
    workers=1,
    sg=0,
    epochs=WORD2VEC_EPOCHS,
    seed=SEED,
)

italian_word2vec_model = Word2Vec(
    sentences=italian_word2vec_sentences,
    vector_size=WORD2VEC_DIMENSION,
    window=WORD2VEC_WINDOW,
    min_count=WORD2VEC_MIN_COUNT,
    workers=1,
    sg=0,
    epochs=WORD2VEC_EPOCHS,
    seed=SEED,
)

word2vec_preparation_seconds = (
    time.perf_counter() - word2vec_start_time
)


english_word2vec_path = (
    MODEL_DIR / "english_word2vec.model"
)
italian_word2vec_path = (
    MODEL_DIR / "italian_word2vec.model"
)

english_word2vec_model.save(
    str(english_word2vec_path)
)
italian_word2vec_model.save(
    str(italian_word2vec_path)
)


def initialise_embedding_from_word2vec(
    embedding_layer,
    token_to_index,
    word2vec_model,
):
    """Copy available Word2Vec vectors into an embedding layer."""

    vocabulary_tokens = [
        token
        for token in token_to_index
        if token not in SPECIAL_TOKENS
    ]

    matched_tokens = 0

    with torch.no_grad():
        for token in vocabulary_tokens:
            if token not in word2vec_model.wv:
                continue

            token_index = token_to_index[token]

            vector = torch.tensor(
                word2vec_model.wv[token],
                dtype=embedding_layer.weight.dtype,
                device=embedding_layer.weight.device,
            )

            embedding_layer.weight[token_index].copy_(vector)
            matched_tokens += 1

    coverage = matched_tokens / len(vocabulary_tokens)

    return matched_tokens, coverage


print(
    f"English Word2Vec vocabulary: "
    f"{len(english_word2vec_model.wv):,}"
)
print(
    f"Italian Word2Vec vocabulary: "
    f"{len(italian_word2vec_model.wv):,}"
)
print(
    f"Preparation time: "
    f"{word2vec_preparation_seconds / 60:.2f} minutes"
)
print(f"Saved English model: {english_word2vec_path}")
print(f"Saved Italian model: {italian_word2vec_path}")

English Word2Vec vocabulary: 17,720
Italian Word2Vec vocabulary: 23,233
Preparation time: 0.28 minutes
Saved English model: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\models\task3\english_word2vec.model
Saved Italian model: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\models\task3\italian_word2vec.model


In [73]:
WORD2VEC_LSTM_NUMBER_OF_EPOCHS = 20


# Recreate the same deterministic training order.
word2vec_training_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(SEED),
)


# Create the same large LSTM architecture.
torch.manual_seed(SEED)

word2vec_lstm_model = create_lstm_model(
    source_vocabulary_size=len(english_token_to_index),
    target_vocabulary_size=len(italian_token_to_index),
    embedding_dimension=BEST_LSTM_EMBEDDING_DIMENSION,
    hidden_dimension=BEST_LSTM_HIDDEN_DIMENSION,
)


# Initialise the encoder embedding with English Word2Vec.
english_matched_tokens, english_embedding_coverage = (
    initialise_embedding_from_word2vec(
        word2vec_lstm_model.encoder.embedding,
        english_token_to_index,
        english_word2vec_model,
    )
)

# Initialise the decoder embedding with Italian Word2Vec.
italian_matched_tokens, italian_embedding_coverage = (
    initialise_embedding_from_word2vec(
        word2vec_lstm_model.decoder.embedding,
        italian_token_to_index,
        italian_word2vec_model,
    )
)


# The embeddings remain trainable.
assert word2vec_lstm_model.encoder.embedding.weight.requires_grad
assert word2vec_lstm_model.decoder.embedding.weight.requires_grad


print(
    f"English embedding coverage: "
    f"{english_matched_tokens:,} tokens "
    f"({english_embedding_coverage:.1%})"
)
print(
    f"Italian embedding coverage: "
    f"{italian_matched_tokens:,} tokens "
    f"({italian_embedding_coverage:.1%})"
)


word2vec_lstm_history = run_experiment(
    model=word2vec_lstm_model,
    training_loader=word2vec_training_loader,
    validation_loader=validation_loader,
    checkpoint_name=(
        "english_to_italian_context_lstm_large_word2vec.pt"
    ),
    number_of_epochs=WORD2VEC_LSTM_NUMBER_OF_EPOCHS,
    learning_rate=BEST_LSTM_LEARNING_RATE,
)


word2vec_validation_references, word2vec_validation_hypotheses = (
    generate_translations(
        word2vec_lstm_model,
        validation_loader,
        italian_index_to_token,
    )
)

word2vec_validation_metrics = compute_translation_metrics(
    word2vec_validation_references,
    word2vec_validation_hypotheses,
    target_language="italian",
)


random_mean_length = np.mean(
    [
        len(hypothesis)
        for hypothesis in large_lstm_validation_hypotheses
    ]
)

word2vec_mean_length = np.mean(
    [
        len(hypothesis)
        for hypothesis in word2vec_validation_hypotheses
    ]
)

random_unique_translations = len(
    set(map(tuple, large_lstm_validation_hypotheses))
)

word2vec_unique_translations = len(
    set(map(tuple, word2vec_validation_hypotheses))
)


embedding_comparison = pd.DataFrame(
    [
        {
            "embedding": "learned_from_scratch",
            "validation_loss": large_lstm_history[
                "best_validation_loss"
            ],
            "BLEU": large_lstm_validation_metrics["BLEU"],
            "METEOR": large_lstm_validation_metrics["METEOR"],
            "mean_generated_length": random_mean_length,
            "unique_translations": random_unique_translations,
            "embedding_preparation_minutes": 0.0,
            "training_minutes": (
                large_lstm_history["elapsed_seconds"] / 60
            ),
        },
        {
            "embedding": "word2vec_initialisation",
            "validation_loss": word2vec_lstm_history[
                "best_validation_loss"
            ],
            "BLEU": word2vec_validation_metrics["BLEU"],
            "METEOR": word2vec_validation_metrics["METEOR"],
            "mean_generated_length": word2vec_mean_length,
            "unique_translations": word2vec_unique_translations,
            "embedding_preparation_minutes": (
                word2vec_preparation_seconds / 60
            ),
            "training_minutes": (
                word2vec_lstm_history["elapsed_seconds"] / 60
            ),
        },
    ]
)

display(
    embedding_comparison.round(
        {
            "validation_loss": 4,
            "BLEU": 2,
            "METEOR": 2,
            "mean_generated_length": 2,
            "embedding_preparation_minutes": 2,
            "training_minutes": 2,
        }
    )
)


embedding_comparison_path = (
    RESULTS_DIR / "embedding_comparison.csv"
)

embedding_comparison.to_csv(
    embedding_comparison_path,
    index=False,
)

print("\nWord2Vec validation results")
print(f"BLEU: {word2vec_validation_metrics['BLEU']:.2f}")
print(f"METEOR: {word2vec_validation_metrics['METEOR']:.2f}")
print(f"Saved comparison: {embedding_comparison_path}")


print("\nWord2Vec translation examples:")
for i in range(5):
    print(f"\nEN:  {validation_pairs[i][0]}")
    print(
        f"REF: "
        f"{' '.join(word2vec_validation_references[i])}"
    )
    print(
        f"HYP: "
        f"{' '.join(word2vec_validation_hypotheses[i]) or '<empty>'}"
    )

English embedding coverage: 9,996 tokens (100.0%)
Italian embedding coverage: 9,996 tokens (100.0%)
Epoch 1/20 | training loss: 5.8190 | validation loss: 4.9788
Epoch 2/20 | training loss: 4.7078 | validation loss: 4.4025
Epoch 3/20 | training loss: 4.2565 | validation loss: 4.1325
Epoch 4/20 | training loss: 3.9764 | validation loss: 3.9562
Epoch 5/20 | training loss: 3.7627 | validation loss: 3.8409
Epoch 6/20 | training loss: 3.5873 | validation loss: 3.7527
Epoch 7/20 | training loss: 3.4351 | validation loss: 3.6848
Epoch 8/20 | training loss: 3.2996 | validation loss: 3.6376
Epoch 9/20 | training loss: 3.1747 | validation loss: 3.6025
Epoch 10/20 | training loss: 3.0588 | validation loss: 3.5803
Epoch 11/20 | training loss: 2.9505 | validation loss: 3.5684
Epoch 12/20 | training loss: 2.8469 | validation loss: 3.5596
Epoch 13/20 | training loss: 2.7496 | validation loss: 3.5653
Epoch 14/20 | training loss: 2.6559 | validation loss: 3.5669
Epoch 15/20 | training loss: 2.5659 | val

,embedding,validation_loss,BLEU,METEOR,mean_generated_length,unique_translations,embedding_preparation_minutes,training_minutes
0,learned_from_scratch,3.7816,3.92,18.23,14.22,6811,0.00,63.87
1,word2vec_initialisation,3.5596,4.17,19.23,14.60,6814,0.28,91.76



Word2Vec validation results
BLEU: 4.17
METEOR: 19.23
Saved comparison: C:\Users\tella\OneDrive\Desktop\progetti NLP\NLP_2\English-Italian-Machine-Translation-\results\task3_outputs\embedding_comparison.csv

Word2Vec translation examples:

EN:  ... which is a fundamental element in the fight against organised crime and its links with terrorism?
REF: ...elemento fondamentale per la lotta al crimine organizzato e ai suoi legami con il terrorismo?
HYP: "tuttavia, è una lotta contro la lotta contro la lotta contro la lotta contro la criminalità e la sua parte.

EN:  "my group was appalled, politically speaking, at the blandness of the terms of the socialist pseudo-censure motion of december."
REF: "il mio gruppo, politicamente parlando, si è stupito della mitezza del testo della pseudomozione di censura socialista presentata in dicembre."
HYP: "la mia relazione è stata presentata dal punto di vista della corte dei conti di cui ho votato a favore della

EN:  "ultimately, the decisions were 